Human In the Loop

Review: For human-in-the-loop, we often want to see our graph outputs as its running.

We laid the foundations for this with streaming.

Goals Now, let's talk about the motivations for human-in-the-loop:

Approval - We can interrupt our agent, surface state to a user, and allow the user to accept an action.
Debugging - We can rewind the graph to reproduce or avoid issues.
Editing - You can modify the state.

LangGraph offers several ways to get or update agent state to support various human-in-the-loop workflows.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen-qwq-32b")
result=llm.invoke("Write a short story about a robot learning to love.")

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BadRequestError: Error code: 400 - {'error': {'message': 'The model `qwen-qwq-32b` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
### Custom tools

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b


def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b


def divide(a: int, b: int) -> int:
    """Divides a by b.

    Args:
        a: first int
        b: second int
    """
    return a // b

tools=[add, multiply, divide]

In [ ]:
llm_with_tools=llm.bind_tools(tools)
llm_with_tools

In [ ]:
### Workflow with LangGraph

from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage

sys_msg=SystemMessage(
    content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
)


## node definition
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# Graph
builder = StateGraph(MessagesState)

## Define nodes:
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

#edges
builder.add_edge(START, "assistant")

builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to "tools"
    # If the latest message (result) from assistant is not a tool call -> tools_condition routes to END
    tools_condition,
)

builder.add_edge("tools", "assistant")

memory = MemorySaver()

#human in the loop
graph = builder.compile(interrupt_before=["assistant"], checkpointer=memory)

In [ ]:
thread = {"configurable": {"thread_id": "123"}}

initial_input = {
    "messages": HumanMessage(content="Multiply 2 and 3")
}

In [ ]:
for event in graph.stream(
    initial_input,
    thread,
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

In [ ]:
state=graph.get_state(thread)
state.next

In [ ]:
state

In [ ]:
for event in graph.stream(
    None,
    thread,
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

In [ ]:
state=graph.get_state(thread)
state.next

In [ ]:
for event in graph.stream(
    None,
    thread,
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

edit human feedback

In [ ]:
state=graph.get_state(thread)

In [ ]:
graph.update_state(thread,{"messages":[HumanMessage(content="No,Divide 10 by 2")]})

new_state=graph.get_state(thread).values()

In [ ]:
for event in graph.stream(
    None,
    thread,
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

In [ ]:
for event in graph.stream(
    None,
    thread,
    stream_mode="values"
):
    event["messages"][-1].pretty_print()

workflow will wait for the human input

In [ ]:
sys_msg=SystemMessage(
    content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
)

In [ ]:
### Human feedback node

def human_feedback(state: MessagesState):
    pass

### Assistant node

def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

## Graph

# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_node("human_feedback", human_feedback)

In [ ]:
## Define the edges

builder.add_edge(START, "human_feedback")
builder.add_edge("human_feedback", "assistant")

builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to "tools"
    # If the latest message (result) from assistant is not a tool call -> tools_condition routes to END
    tools_condition,
)

builder.add_edge("tools", "human_feedback")

In [ ]:
memory = MemorySaver()

graph = builder.compile(
    interrupt_before=["human_feedback"],
    checkpointer=memory
)

In [ ]:
# Input
initial_input = {"messages": "Multiply 2 and 3"}

# Thread
thread = {"configurable": {"thread_id": "5"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

## get user input

user_input = input("Tell me how you want to update the state: ")
graph.update_state(
    thread,
    {"messages": user_input},
    as_node="human_feedback"
)

# Continue the graph execution
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

In [ ]:

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()